# Motion Planning Gadgets — Interactive Guide

This notebook is the primary reference for the Python gadget library.  Run cells top-to-bottom; each section is self-contained once the setup cell has run.

**Contents**
1. Setup
2. Creating Gadgets
3. Tracing Traversal Sequences
4. Named States & Locations in Visualizations
5. Closure & Idempotency
6. Simulation Checker
7. Quick Reference

---

## 1. Setup

Run this cell once.  All modules live in the same `src/` directory as this notebook, so no installation is needed beyond `graphviz` and `IPython` (both come with Jupyter).

In [ ]:
from gadget import Gadget
from gadget_simulation import (
    get_traversal_modes,
    behaviorally_equivalent,
    refute_simulation,
    is_reversible,
    is_dag,
)
from gadget_system import GadgetSystem
from gadget_notebook import show, show_comparison

print('Ready.')

---
## 2. Creating Gadgets

A `Gadget` has **states** (internal memory) and **locations** (the ports the robot can enter/exit).  A transition `(from_state, from_loc, to_state, to_loc)` means:

> When the gadget is in `from_state`, the robot may enter at `from_loc` and exit at `to_loc`, leaving the gadget in `to_state`.

Locations must differ (`from_loc ≠ to_loc`).  Setting `reversible=True` automatically adds the reverse of each transition.

In [ ]:
# Antiparallel 2-toggle: 2 states, 2 locations.
# State 0 (open):   robot may enter loc 0 and exit loc 1, toggling to state 1.
# State 1 (closed): robot may enter loc 1 and exit loc 0, toggling to state 0.
ap2t = Gadget(num_states=2, num_locations=2)
ap2t.add_transition(0, 0, 1, 1)   # open:   in -> out, now closed
ap2t.add_transition(1, 1, 0, 0)   # closed: out -> in, now open

print('Transitions:', ap2t.transitions)
print('Deterministic:', ap2t.is_deterministic())

In [ ]:
# Reversible chain: 4 states, 1 location.
# The robot can move freely along the chain in either direction.
chain = Gadget(num_states=4, num_locations=1, reversible=True)
chain.add_transition(0, 0, 1, 0)
chain.add_transition(1, 0, 2, 0)
chain.add_transition(2, 0, 3, 0)

print(f'Transitions (including auto-added reverses): {len(chain.transitions)}')
for t in chain.transitions:
    print(' ', t)

---
## 3. Tracing Traversal Sequences

`gadget.trace(initial_state, moves)` simulates a robot passing through the gadget and prints each step.  A **move** is an `(entry_loc, exit_loc)` pair.

Use `trace` to:
- Verify a construction behaves as intended for each claimed traversal mode.
- Confirm that disallowed traversals indeed fail.
- Produce step-by-step evidence for a write-up.

Optional `state_labels` and `loc_labels` dicts replace numeric indices with readable names.

In [ ]:
# Verify both traversal modes of the antiparallel toggle.
s = {0: 'open', 1: 'closed'}
l = {0: 'in',   1: 'out'}

print('=== Mode 1: open state, enter in, exit out ===')
ap2t.trace(initial_state=0, moves=[(0, 1)], state_labels=s, loc_labels=l)

print()
print('=== Mode 2: closed state, enter out, exit in ===')
ap2t.trace(initial_state=1, moves=[(1, 0)], state_labels=s, loc_labels=l)

In [ ]:
# Confirm that a disallowed traversal fails with a helpful message.
print('=== Disallowed: open state, enter in, exit in ===')
ap2t.trace(initial_state=0, moves=[(0, 0)], state_labels=s, loc_labels=l)

In [ ]:
# Multi-step trace: robot uses the gadget twice in a row.
print('=== Two traversals: open -> closed -> open ===')
ap2t.trace(initial_state=0, moves=[(0, 1), (1, 0)], state_labels=s, loc_labels=l)

---
## 4. Named States & Locations in Visualizations

All three visualizers (`visualize_gadget`, `visualize_gadget_comparison`,
`visualize_gadget_circular`) accept optional `state_labels` and `loc_labels`
keyword arguments.  In this notebook use `show()` and `show_comparison()` from
`gadget_notebook` — they display the circular layout inline without leaving
temporary files on disk.

```python
show(gadget, title='', state_labels={...}, loc_labels={...})
show_comparison(g1, g2, title='', state_labels={...}, loc_labels={...})
```

For saving to a file use the underlying functions directly:

```python
from gadget_viz import visualize_gadget_circular
visualize_gadget_circular(gadget, 'output_path', format='png',
                          state_labels={0: 'open', 1: 'closed'},
                          loc_labels={0: 'in', 1: 'out'})
```

In [ ]:
show(ap2t, title='Antiparallel 2-toggle',
     state_labels={0: 'open', 1: 'closed'},
     loc_labels={0: 'in', 1: 'out'})

In [ ]:
# A locking 2-toggle: 3 states, 4 locations.
l2t = Gadget(3, 4, reversible=True)
l2t.add_transition(0, 0, 1, 1)   # traverse left corridor: open -> half
l2t.add_transition(0, 2, 2, 3)   # traverse right corridor: open -> locked

show(l2t, title='Locking 2-toggle',
     state_labels={0: 'open', 1: 'half', 2: 'locked'},
     loc_labels={0: 'L-in', 1: 'L-out', 2: 'R-in', 3: 'R-out'})

---
## 5. Closure & Idempotency

`gadget.closure()` returns a new gadget with all **shortcut transitions** added:
if a robot can enter at location `l` in state `s`, navigate internally, and exit at
location `l'` in state `s'`, then `(s, l, s', l')` appears in the closure even if it
required intermediate steps.

**Idempotency check** — the closure of the closure must equal the closure:
```python
set(g.closure().closure().transitions) == set(g.closure().transitions)
```
This is a good sanity check after defining a new gadget.

In [ ]:
# 3-state chain through location 0.
chain3 = Gadget(3, 2)
chain3.add_transition(0, 0, 1, 1)   # state 0 -> state 1 via loc 0
chain3.add_transition(1, 0, 2, 1)   # state 1 -> state 2 via loc 0

closed = chain3.closure()

print('Original transitions:')
for t in sorted(chain3.transitions): print(' ', t)

print()
print('After closure (shortcuts added):')
for t in sorted(closed.transitions): print(' ', t)

print()
print('Idempotency check:',
      set(closed.closure().transitions) == set(closed.transitions))

In [ ]:
show_comparison(chain3, closed,
                title='Chain gadget: original (left) vs. closure (right)',
                state_labels={0: 'A', 1: 'B', 2: 'C'},
                loc_labels={0: 'entry', 1: 'exit'})

In [ ]:
# Closure of a reversible gadget — shortcuts are added in both directions.
rev3 = Gadget(3, 2, reversible=True)
rev3.add_transition(0, 0, 1, 1)
rev3.add_transition(1, 0, 2, 1)

rev3_closed = rev3.closure()
print('Reversible closure transitions:')
for t in sorted(rev3_closed.transitions): print(' ', t)

# Spot-check the reverse shortcut: enter loc 1 in state 2, exit loc 0 in state 0
assert (2, 1, 0, 0) in rev3_closed.transitions, 'reverse shortcut missing!'
print()
print('Reverse shortcut (2,1,0,0) present:', (2, 1, 0, 0) in rev3_closed.transitions)

---
## 6. Equivalence and Simulation

Two distinct questions, kept separate:

- **Equivalence** — are two gadgets the *same gadget* (do they admit exactly the
  same traversal sequences)?  Decided by `behaviorally_equivalent(A, B)`, which
  determinises, minimises, and canonicalises each gadget before comparing.
- **Simulation** — can a *system* of one gadget reproduce another?  We *refute*
  it soundly with `refute_simulation`, and *confirm* it by building an explicit
  construction with `GadgetSystem` (Section 6c).

> The old `check_simulation` was removed: comparing single-step traversal modes
> is neither necessary nor sufficient for simulation — *extra* reachable
> behaviour breaks a simulation rather than helping it.

In [ ]:
# A toggle and a dicrumbler are NOT the same gadget.
toggle = Gadget(2, 2); toggle.add_transition(0, 0, 1, 1); toggle.add_transition(1, 1, 0, 0)
dicrumbler = Gadget(2, 2); dicrumbler.add_transition(0, 0, 1, 1)
print('toggle == dicrumbler         ?', behaviorally_equivalent(toggle, dicrumbler))

# A state-relabelled toggle IS the same gadget.
toggle2 = Gadget(2, 2); toggle2.add_transition(1, 0, 0, 1); toggle2.add_transition(0, 1, 1, 0)
print('toggle == relabelled toggle  ?', behaviorally_equivalent(toggle, toggle2))

### 6a. Refuting a simulation

`refute_simulation(target, block)` returns a list of *sound* reasons that no
system of `block` can simulate `target` (using invariants preserved under wiring:
reversibility and DAG/boundedness).  An empty list means "not refuted" — which is
**not** a proof that a simulation exists.

In [ ]:
# A toggle is reversible, a dicrumbler is not; a dicrumbler is a DAG, a toggle is not.
print('can toggles      simulate a dicrumbler?', refute_simulation(dicrumbler, toggle, verbose=False) or 'not refuted')
print('can dicrumblers  simulate a toggle?    ', refute_simulation(toggle, dicrumbler, verbose=False) or 'not refuted')

### 6b. Building and verifying a construction

A `GadgetSystem` is a set of gadget instances wired together at their ports, with
some ports `expose`d as the external interface.  `verify_simulates` computes the
gadget the system *induces* and checks it against a target; `print_all_traces`
shows the agent's path for each allowed traversal.

In [ ]:
# Two toggles in series simulate a single toggle.
sysm = GadgetSystem()
sysm.add_instance('T1', toggle, 0).add_instance('T2', toggle, 0)
sysm.connect(('T1', 1), ('T2', 0))
sysm.expose('L', ('T1', 0)).expose('R', ('T2', 1))

print('simulates a toggle?',
      sysm.verify_simulates(toggle, label_to_target_loc={'L': 0, 'R': 1}, verbose=False))
print()
sysm.print_all_traces()

See `src/demo_simulations.py` and `demo_output/REPORT.md` for many more
worked constructions (including multi-port gadgets and branching networks), and
`visualize_system` / `visualize_gadget_box` for paper-style diagrams.

---
## 7. Quick Reference

### Gadget construction
```python
g = Gadget(num_states, num_locations, reversible=False)
g.add_transition(from_state, from_loc, to_state, to_loc)  # returns False if invalid
g.get_transitions_from(state, loc)   # -> [(to_state, to_loc), ...]
g.closure()                          # -> new Gadget with all shortcut transitions
```

### Tracing a single gadget
```python
ok = g.trace(initial_state, moves,   # moves = [(entry_loc, exit_loc), ...]
             state_labels={0: 'open', 1: 'closed'}, loc_labels={0: 'in', 1: 'out'})
```

### Equivalence & simulation (`gadget_simulation.py`)
```python
behaviorally_equivalent(A, B, fixed_locations=False)   # are A and B the same gadget?
is_reversible(g);  is_dag(g)                            # composition invariants
refute_simulation(target, block)                       # sound impossibility check
```

### Constructions (`gadget_system.py`)
```python
s = GadgetSystem()
s.add_instance('T1', toggle, initial_state=0)
s.connect(('T1', 1), ('T2', 0))            # wire ports together
s.expose('L', ('T1', 0))                   # expose an external interface port
s.verify_simulates(target, label_to_target_loc={'L': 0, 'R': 1})
s.print_all_traces()                       # agent path for each traversal
```

### Visualization
```python
from gadget_notebook import show, show_comparison          # inline (notebook)
from gadget_viz import (visualize_gadget_box,              # paper-style box
                        visualize_state_diagram,           # states + transitions
                        visualize_state_location_diagram,  # configuration graph
                        visualize_system)                  # network + trace overlay
```